# Orbit and coverage diagnostics

This notebook moves from v0.13 diagnostics to v0.14 read-only orbit reports and v0.15 materialized orbit batches.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np

from notebooks._tutorial_utils import plot_coverage_counts, pretty_json
from pdelie.data import generate_heat_1d_field_batch
from pdelie.derivatives import compute_spectral_fd_derivatives
from pdelie.invariants import (
    build_uniform_translation_orbit_batch,
    compute_periodic_window_coverage,
    diagnose_uniform_translation_consistency,
    summarize_uniform_translation_orbit,
)
from pdelie.reporting import summarize_residual_batch
from pdelie.residuals import HeatResidualEvaluator

DOMAIN_LENGTH = 2.0 * np.pi
CONFIG = {
    "shifts": [0.0, DOMAIN_LENGTH / 64.0, DOMAIN_LENGTH / 8.0, -DOMAIN_LENGTH / 8.0, DOMAIN_LENGTH / 8.0, DOMAIN_LENGTH],
    "windows": [{"start": 0.0, "width": DOMAIN_LENGTH / 4.0}],
}
CONFIG


## 1. Coverage is grid-point coverage under a fixed convention

Positive shift means: translate the field, then observe a fixed window. In original coordinates, that is the preimage of the fixed window.


In [ ]:
x = np.linspace(0.0, DOMAIN_LENGTH, 64, endpoint=False)
coverage = compute_periodic_window_coverage(x=x, windows=CONFIG["windows"], shifts=CONFIG["shifts"])
print(pretty_json({
    "coverage_convention": coverage["coverage_convention"],
    "shift_convention": coverage["shift_convention"],
    "coverage_fraction": coverage["coverage_fraction"],
    "duplicate_shifts_count": len(CONFIG["shifts"]),
}))
plot_coverage_counts(coverage, title="Coverage counts with duplicate shift preserved")


## 2. Translation consistency is report-only

This diagnostic creates transformed fields internally, but returns a JSON-compatible report instead of a transformed dataset.


In [ ]:
field = generate_heat_1d_field_batch(batch_size=2, num_times=17, num_points=64, seed=660)
evaluator = HeatResidualEvaluator()
consistency = diagnose_uniform_translation_consistency(field, shifts=CONFIG["shifts"], residual_evaluator=evaluator)
print(pretty_json({
    "summary_type": consistency["summary_type"],
    "shift_count": len(consistency["shift_reports"]),
    "all_residual_stability_passed": all(report["residual_stability_passed"] for report in consistency["shift_reports"]),
}, max_chars=2500))


## 3. A read-only orbit report combines coverage and consistency

`summary_uniform_translation_orbit(...)` supports workflow reporting without constructing augmented data.


In [ ]:
orbit_report = summarize_uniform_translation_orbit(
    field,
    shifts=CONFIG["shifts"],
    windows=CONFIG["windows"],
    residual_evaluator=evaluator,
    source_field_id="heat_seed_660",
)
print(pretty_json({
    "summary_type": orbit_report["summary_type"],
    "orbit_passed": orbit_report["orbit_passed"],
    "source_field_id": orbit_report["source_field_id"],
    "coverage_fraction": orbit_report["coverage"]["coverage_fraction"],
}))


## 4. Materialized orbit batches are explicit data utilities

Use orbit batches when a downstream method needs actual transformed samples. Keep source and shift indices enabled for auditability.


In [ ]:
result = build_uniform_translation_orbit_batch(
    field,
    shifts=CONFIG["shifts"],
    source_field_id="heat_seed_660",
)
residual = evaluator.evaluate(result.field, compute_spectral_fd_derivatives(result.field))
residual_summary = summarize_residual_batch(residual)
print(pretty_json({
    "source_shape": list(field.values.shape),
    "orbit_shape": list(result.field.values.shape),
    "ordering": result.report["ordering"],
    "source_indices_head": result.report["source_batch_indices"][:10],
    "shift_indices_head": result.report["shift_indices"][:10],
    "residual_rms": residual_summary["rms_residual"],
    "split_policy_warning": "orbit batches do not decide train/heldout policy",
}, max_chars=3500))


## Takeaway

Use read-only reports to audit transformations. Use materialized orbit batches only when a downstream method needs transformed data, and keep provenance attached.
